# Module 3 — Lab Bài 1: LLM có làm được việc này không?
## Notebook sinh viên · Google Colab · 90 phút

**Mục tiêu:** thử → đo → nhìn lỗi → quyết định có nên dùng LLM cho một công việc. Không fine-tune, không RAG, không triển khai hệ thống.

Notebook có sẵn phần lớn code. Sinh viên chỉnh prompt/tham số/dữ liệu và bổ sung nhận xét tại các ô **THỰC HÀNH**. Dữ liệu là 15 ticket tiếng Việt hoàn toàn mô phỏng.

Dùng Gemini Developer API từ Google AI Studio, project free tier. Các cell setup tách rõ API key, model/URL, metadata và lần gọi thử. Notebook chỉ có một đường gọi API để dễ theo dõi; cần internet và key hợp lệ.

| Phút | Phần |
|---|---|
| 0–10 | 1. Business Context |
| 10–20 | 2. Input / Data và 3. AI Black Box |
| 20–50 | 4. Experiment A/B/C/D |
| 50–65 | Prompt, tham số và missing-data test |
| 65–80 | 5. Evaluation |
| 80–90 | 6. Business Reflection và nộp |

**Nộp:** notebook có output; bảng đánh giá đủ 4 task; ít nhất 1 failure case có input/output/bằng chứng; 3 nguyên tắc cá nhân khi không nên dùng LLM.

**Nhóm:** … · **Thành viên:** … · **Model:** …


# 1. Business Context
### Bước 1 — Công việc cần hỗ trợ
Một nhóm vận hành nhận ticket sự cố tiếng Việt. Kỹ thuật viên cần đọc nhanh tình hình, chuyển đúng nhóm xử lý, trích thông tin vào hệ thống và chọn bước kiểm tra tiếp theo.

LLM chỉ tạo nội dung hỗ trợ. Notebook không thực hiện lệnh trên thiết bị hoặc thay đổi cấu hình.

**THỰC HÀNH — điền:**
- Người dùng kết quả: …
- Pain point cụ thể và cách hiện tại giải quyết: …
- Quyết định được hỗ trợ: …
- Nếu model thêm một nguyên nhân không có bằng chứng, hậu quả là gì? …
- Đầu ra nào cần con người kiểm tra trước khi sử dụng? …

**Câu hỏi gợi ý:** tác vụ nào có thể làm bằng template/quy tắc đơn giản thay vì LLM?


# 2. Input / Data
### Bước 2 — Imports và dữ liệu nhỏ
Colab thường có `requests`, `pandas` và `IPython`. Nếu môi trường local thiếu, chạy `%pip install requests pandas` trong một cell riêng rồi import lại. Notebook dùng REST với `requests` để thấy rõ URL và JSON payload, không cần SDK/GPU.


In [1]:
import os, json, time
from pathlib import Path
from collections import Counter
import requests
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
OUTPUT_DIR = Path('lab1_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)
LABELS = ['Connectivity', 'Performance', 'Hardware', 'Configuration', 'Other']


In [2]:
# Tất cả incident, site và thời gian dưới đây đều là dữ liệu mô phỏng.
INCIDENTS = [
 {'ticket_id':'T01','site_id':'HN01','severity':'high','time':'2026-01-10 20:05',
  'text': 'Site HN01 ghi nhận mất kết nối từ 20:05 ngày 10/01/2026, severity cao. Khoảng 120 người dùng không truy cập được dịch vụ. Ping tới gateway thất bại.\nLúc 20:15 kỹ thuật viên kiểm tra cáp và nguồn: chưa thấy dấu hiệu bất thường. Lúc 20:25 vẫn không truy cập được gateway; đã mở ticket với nhà cung cấp đường truyền.\nĐường truyền phía nhà cung cấp được nghi ngờ, nhưng chưa có log hoặc xác nhận nguyên nhân. Đến 20:45 dịch vụ vẫn chưa khôi phục; đang chờ phản hồi nhà cung cấp.',
  'label':'Connectivity','ambiguous':False},
 {'ticket_id':'T02','site_id':'DN02','severity':'medium','time':'2026-01-11 09:00',
  'text':'Site DN02 lúc 09:00 ngày 11/01/2026 vẫn kết nối được nhưng độ trễ tăng từ 20 ms lên 350 ms; tải file chậm. Severity trung bình. Chưa xác định nguyên nhân.','label':'Performance','ambiguous':False},
 {'ticket_id':'T03','site_id':'HCM03','severity':'high','time':'2026-01-11 10:00',
  'text':'Site HCM03 lúc 10:00 ngày 11/01/2026 phát hiện quạt switch không quay và đèn báo lỗi quạt. Severity cao. Chưa thay linh kiện.','label':'Hardware','ambiguous':False},
 {'ticket_id':'T04','site_id':'HN04','severity':'medium','time':'2026-01-11 11:00',
  'text':'Site HN04 lúc 11:00 ngày 11/01/2026 kiểm tra thấy VLAN của cổng người dùng khác với VLAN trong cấu hình được phê duyệt. Severity trung bình. Ticket yêu cầu kiểm tra sai lệch cấu hình, chưa thay đổi cổng.','label':'Configuration','ambiguous':False},
 {'ticket_id':'T05','site_id':'CT05','severity':'low','time':None,
  'text':'Site CT05 yêu cầu bổ sung thông tin liên hệ vào hồ sơ bàn giao. Severity thấp. Không báo sự cố kỹ thuật, không cung cấp thời gian.','label':'Other','ambiguous':False},
 {'ticket_id':'T06','site_id':'HP06','severity':'high','time':'2026-01-11 13:00',
  'text':'Site HP06 không truy cập được gateway từ 13:00 ngày 11/01/2026. Severity cao. Kỹ thuật viên ghi nhận nhiều lần timeout; chưa xác nhận lỗi phần cứng.','label':'Connectivity','ambiguous':False},
 {'ticket_id':'T07','site_id':'DN07','severity':'medium','time':None,
  'text':'Site DN07: người dùng vẫn truy cập được, nhưng throughput chỉ bằng một phần tư mức thường ngày. Severity trung bình. Chưa có thời điểm bắt đầu hoặc nguyên nhân.','label':'Performance','ambiguous':False},
 {'ticket_id':'T08','site_id':'HCM08','severity':'high','time':None,
  'text':'Site HCM08: bộ nguồn dự phòng có đèn báo lỗi; đo kiểm xác nhận bộ nguồn dự phòng không cấp điện. Severity cao. Bộ nguồn chính vẫn hoạt động; chưa thay linh kiện.','label':'Hardware','ambiguous':False},
 {'ticket_id':'T09','site_id':None,'severity':None,'time':None,
  'text':'Kỹ thuật viên báo một cổng switch có chân tiếp xúc bị gãy khi kiểm tra trực tiếp. Chưa cung cấp mã site, severity, thời gian hoặc nguyên nhân gây hư hỏng.','label':'Hardware','ambiguous':False},
 {'ticket_id':'T10','site_id':'HN10','severity':'medium','time':None,
  'text':'Site HN10: DNS server trong cấu hình máy khách khác với giá trị được phê duyệt. Severity trung bình. Chưa chỉnh cấu hình; yêu cầu xác minh sai lệch.','label':'Configuration','ambiguous':False},
 {'ticket_id':'T11','site_id':'BD11','severity':'low','time':None,
  'text':'Site BD11 đề nghị gửi lại tài liệu hướng dẫn tiếp nhận ticket cho nhân viên mới. Severity thấp. Không có incident kỹ thuật.','label':'Other','ambiguous':False},
 {'ticket_id':'T12','site_id':'QN12','severity':'high','time':None,
  'text':'Site QN12: toàn bộ người dùng không kết nối được mạng; ping gateway thất bại. Severity cao. Chưa thu thập thêm dữ liệu.','label':'Connectivity','ambiguous':False},
 {'ticket_id':'T13','site_id':'HN13','severity':'medium','time':None,
  'text':'Site HN13: người dùng kết nối được nhưng phản hồi ứng dụng chậm và jitter tăng. Severity trung bình. Chưa có log nguyên nhân.','label':'Performance','ambiguous':False},
 {'ticket_id':'T14','site_id':'HCM14','severity':None,'time':None,
  'text':'Site HCM14 có hiện tượng chập chờn sau một lần bảo trì; có lúc timeout, có lúc phản hồi chậm. Chưa có log, chưa biết có thay cấu hình hay linh kiện không.','label':'Other','ambiguous':True},
 {'ticket_id':'T15','site_id':'DN15','severity':None,'time':None,
  'text':'Site DN15 báo dịch vụ không ổn định. Ticket chưa mô tả triệu chứng cụ thể, thời gian hoặc phạm vi ảnh hưởng.','label':'Other','ambiguous':True}
]
tickets = pd.DataFrame(INCIDENTS)
BY_ID = {r['ticket_id']: r for r in INCIDENTS}
assert 10 <= len(tickets) <= 20
assert tickets['ticket_id'].is_unique and tickets['label'].isin(LABELS).all()
display(tickets[['ticket_id','site_id','severity','time','label','ambiguous']])
print(BY_ID['T01']['text'])
tickets.to_csv(OUTPUT_DIR/'tickets_simulated.csv', index=False)


,ticket_id,site_id,severity,time,label,ambiguous
0,T01,HN01,high,2026-01-10 20:05,Connectivity,False
1,T02,DN02,medium,2026-01-11 09:00,Performance,False
2,T03,HCM03,high,2026-01-11 10:00,Hardware,False
3,T04,HN04,medium,2026-01-11 11:00,Configuration,False
4,T05,CT05,low,None,Other,False
5,T06,HP06,high,2026-01-11 13:00,Connectivity,False
6,T07,DN07,medium,None,Performance,False
7,T08,HCM08,high,None,Hardware,False
8,T09,None,None,None,Hardware,False
9,T10,HN10,medium,None,Configuration,False


Site HN01 ghi nhận mất kết nối từ 20:05 ngày 10/01/2026, severity cao. Khoảng 120 người dùng không truy cập được dịch vụ. Ping tới gateway thất bại.
Lúc 20:15 kỹ thuật viên kiểm tra cáp và nguồn: chưa thấy dấu hiệu bất thường. Lúc 20:25 vẫn không truy cập được gateway; đã mở ticket với nhà cung cấp đường truyền.
Đường truyền phía nhà cung cấp được nghi ngờ, nhưng chưa có log hoặc xác nhận nguyên nhân. Đến 20:45 dịch vụ vẫn chưa khôi phục; đang chờ phản hồi nhà cung cấp.


### Bước 3 — Đọc nhãn và dữ liệu thiếu
Ground truth là nhãn giảng dạy được tác giả gán, không phải sự thật tuyệt đối của dữ liệu nghiệp vụ. T14/T15 được gán Other theo quy tắc: thiếu bằng chứng để xác định nhóm lỗi chính. T14 có triệu chứng chồng lấn, nên một nhãn khác cũng cần được thảo luận thay vì mặc định model sai hoàn toàn.

**Quan trọng:** prompt chỉ gửi mô tả ticket; không gửi cột label, severity/site/time metadata để vô tình tiết lộ đáp án hoặc điền dữ liệu không có trong text.

**THỰC HÀNH:** đọc T09, T14, T15; ghi điều gì có thể biết và điều gì chưa thể kết luận. …


In [3]:
display(tickets['label'].value_counts().to_frame('count'))
print('Metadata thiếu:')
display(tickets[['site_id','severity','time']].isna().sum().to_frame('missing_count'))
for tid in ['T09','T14','T15']:
    print(tid, BY_ID[tid]['text'])


,count
label,
Other,4
Connectivity,3
Performance,3
Hardware,3
Configuration,2


Metadata thiếu:


,missing_count
site_id,1
severity,3
time,10


T09 Kỹ thuật viên báo một cổng switch có chân tiếp xúc bị gãy khi kiểm tra trực tiếp. Chưa cung cấp mã site, severity, thời gian hoặc nguyên nhân gây hư hỏng.
T14 Site HCM14 có hiện tượng chập chờn sau một lần bảo trì; có lúc timeout, có lúc phản hồi chậm. Chưa có log, chưa biết có thay cấu hình hay linh kiện không.
T15 Site DN15 báo dịch vụ không ổn định. Ticket chưa mô tả triệu chứng cụ thể, thời gian hoặc phạm vi ảnh hưởng.


# 3. AI Black Box
### Bước 4a — API key, model và URL
**Đường đi:** Python trong Colab → HTTP POST → model trên máy chủ Google → JSON response → text.

Khi dùng API, “setup model” là chọn model ID và cấu hình request. Không cần tải trọng số, gọi `from_pretrained`, `model.to(device)` hay bật GPU.

1. Tạo key tại [Google AI Studio](https://aistudio.google.com/apikey), dùng project **Free tier**, không bật billing cho lab này.
2. Trong Colab mở **Secrets** (biểu tượng chìa khóa), thêm `GEMINI_API_KEY` và bật quyền truy cập notebook. Local: đặt biến môi trường cùng tên.
3. Chạy cell bên dưới. Không paste key vào code và không in key.

Model mặc định `gemini-3.5-flash-lite` được trang pricing liệt kê free tier cho input/output text (kiểm tra ngày 07/10/2026). Google hiện giới hạn dòng 2.5 cho người đã dùng trước đây, nên không chọn 2.5 làm mặc định cho lớp mới.

Free tier vẫn có quota. Model xuất hiện trong API không chứng minh project có quota miễn phí; kiểm tra **Free tier** và quota trong AI Studio. Key của project có billing có thể phát sinh phí. Chỉ gửi ticket mô phỏng: nội dung free tier có thể được dùng cải thiện sản phẩm.


In [5]:
# 1. Đọc key: ưu tiên biến môi trường, sau đó Colab Secrets.
API_KEY = os.getenv('viettel')
if not API_KEY:
    from google.colab import userdata
    API_KEY = userdata.get('viettel')
assert API_KEY, 'Thêm GEMINI_API_KEY và chạy lại cell này.'

# 2. Chọn model và ghép URL của Gemini Developer API.
MODEL_NAME = 'gemini-3.5-flash-lite'
BASE_URL = 'https://generativelanguage.googleapis.com/v1beta'
MODEL_URL = f'{BASE_URL}/models/{MODEL_NAME}'
API_URL = f'{MODEL_URL}:generateContent'
HEADERS = {'x-goog-api-key': API_KEY, 'Content-Type': 'application/json'}

# 3. Cấu hình thí nghiệm.
LOW_VALUE, HIGH_VALUE = 0.1, 0.9  # temperature
MAX_OUTPUT_TOKENS = 2048
REQUEST_PAUSE_SECONDS = 13      # tăng nếu quota theo phút thấp hơn
CALL_LOG = []
print('Model:', MODEL_NAME)
print('POST URL:', API_URL)       # URL không chứa key


Model: gemini-3.5-flash-lite
POST URL: https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash-lite:generateContent


### Bước 4b — Kiểm tra model trước khi gọi
`GET /models/{model}` trả metadata: phương thức hỗ trợ, token limit và tham số mặc định. Đây là kiểm tra cấu hình, chưa tạo câu trả lời và chưa chứng minh request sinh text sẽ thành công.

Cell sau dùng chính key của bạn để kiểm tra model. Nếu HTTP 404, kiểm tra model ID/quyền truy cập trong AI Studio; sửa `MODEL_NAME` rồi chạy lại **4a và 4b**. Không tự chuyển sang model khác giữa thí nghiệm.


In [6]:
model_response = requests.get(MODEL_URL, headers=HEADERS, timeout=30)
if model_response.status_code != 200:
    raise RuntimeError(f'Kiểm tra model: HTTP {model_response.status_code}. Xem hướng dẫn xử lý lỗi bên dưới.')
model_info = model_response.json()
assert 'generateContent' in model_info.get('supportedGenerationMethods', []), 'Model không hỗ trợ generateContent.'
print('Model được API nhận diện:', model_info['name'])
print('Output token limit:', model_info.get('outputTokenLimit'))
print('Temperature mặc định:', model_info.get('temperature'))
print('Temperature tối đa:', model_info.get('maxTemperature'))


Model được API nhận diện: models/gemini-3.5-flash-lite
Output token limit: 65536
Temperature mặc định: 1
Temperature tối đa: 2


### Bước 5a — Hàm gọi API: chỉ có 3 phần
1. **Payload:** input nằm trong `contents`; tham số nằm trong `generationConfig`.
2. **Gửi:** `requests.post(URL, headers=..., json=...)`.
3. **Đọc:** lấy text từ `candidates[0].content.parts`.

| Trường REST | Ý nghĩa | Dùng trong lab |
|---|---|---|
| `temperature` | Điều khiển độ ngẫu nhiên; API mô tả khoảng 0–2, giới hạn cụ thể xem metadata model | So sánh 0.1 và 0.9 |
| `maxOutputTokens` | Giới hạn token response; không phải số từ | 2048 |
| `topP` | Nucleus sampling | Không gửi, giữ mặc định model |

Tham số REST dùng camelCase: `maxOutputTokens`, `topP`. SDK Python có tên khác như `max_output_tokens`, `top_p`; không trộn hai cách viết. Free tier dùng cùng endpoint và cấu trúc payload; không có `free=True` hay một URL “free API” riêng.

Để phép thử dễ đọc, lab chỉ đổi temperature. Giữ nguyên model, token limit và các tham số khác; output khác nhau không tự chứng minh chất lượng tốt/xấu. Không cấu hình tools, grounding, cache hay JSON mode trong hàm này; task C đánh giá mức tuân thủ prompt JSON thô.


In [7]:
def ask_gemini(prompt, temperature=0.1, max_output_tokens=MAX_OUTPUT_TOKENS):
    # A. Input và tham số
    payload = {
        'contents': [{'role': 'user', 'parts': [{'text': prompt}]}],
        'generationConfig': {
            'temperature': temperature,
            'maxOutputTokens': max_output_tokens,
        },
    }

    # B. Gửi request; chờ giữa các lượt để giảm lỗi quota theo phút.
    time.sleep(REQUEST_PAUSE_SECONDS)
    response = requests.post(API_URL, headers=HEADERS, json=payload, timeout=60)
    if response.status_code != 200:
        raise RuntimeError(f'Gemini HTTP {response.status_code}. Xem bảng xử lý lỗi; không tự retry.')

    # C. Đọc response
    data = response.json()
    if not data.get('candidates'):
        raise RuntimeError('Không có candidate; kiểm tra promptFeedback/safety trong AI Studio.')
    candidate = data['candidates'][0]
    parts = candidate.get('content', {}).get('parts', [])
    text = ''.join(p.get('text', '') for p in parts if not p.get('thought', False))
    if not text.strip():
        raise RuntimeError('Không có text; kiểm tra safety hoặc token limit.')
    if candidate.get('finishReason') == 'MAX_TOKENS':
        print('Cảnh báo: output bị cắt. Ghi nhận trước khi tăng token limit và thử lại.')
    return text


### Bước 5b — Gọi thử một câu ngắn
Một POST thành công mới kiểm tra được key, quyền model, URL và tham số cho project của bạn. Tác giả đã đối chiếu cấu trúc với tài liệu chính thức và kiểm tra bằng response giả lập; chưa gọi API thật vì không có key của lớp.

| HTTP | Việc cần kiểm tra |
|---|---|
| 400 | API key hợp lệ, payload và tên/giá trị tham số; xem chi tiết trong AI Studio |
| 403 | Quyền key/project và khả dụng dịch vụ |
| 404 | Model ID và quyền dùng model |
| 429 | Quota theo phút/token/ngày trong AI Studio; tăng thời gian chờ chỉ giúp nếu chưa hết quota ngày |
| 500 / 503 | Thử lại sau; không chạy vòng retry liên tục |

Khoảng 18 POST cho code có sẵn (bao gồm gọi thử), cộng 1 GET metadata; chạy lại cell sẽ thêm request. Kiểm tra quota trước buổi học. Không hardcode con số quota vì Google có thể thay đổi theo model/project.


In [8]:
print(ask_gemini('Trả lời bằng tiếng Việt trong một câu: LLM là gì?', temperature=0.1, max_output_tokens=256))


LLM (Mô hình ngôn ngữ lớn) là một hệ thống trí tuệ nhân tạo được huấn luyện trên khối lượng dữ liệu văn bản khổng lồ để hiểu, xử lý và tạo ra ngôn ngữ tự nhiên giống con người.


### Bước 5c — Ghép prompt với ticket và lưu bằng chứng
Hàm dưới chỉ giúp các task dùng chung một cách gọi: lấy text ticket → ghép prompt → gọi `ask_gemini` → lưu prompt, temperature, output. Nhãn tham chiếu không được gửi cho model.

`repeat_id` chỉ ghi nhận số lần thử trong báo cáo; không được gửi làm seed và không tạo tính tái lập. Chạy lại cell tạo request mới. Notebook lưu kết quả của phiên hiện tại.


In [9]:
def call_llm(task, ticket_id, prompt, value=LOW_VALUE, repeat_id=0, text_override=None):
    text = text_override if text_override is not None else BY_ID[ticket_id]['text']
    full_prompt = prompt + '\n\n<incident>\n' + text + '\n</incident>'
    answer = ask_gemini(full_prompt, temperature=value)
    record = {
        'task': task, 'ticket_id': ticket_id, 'model': MODEL_NAME,
        'prompt': full_prompt, 'params': {'temperature': value},
        'max_output_tokens': MAX_OUTPUT_TOKENS, 'repeat_id': repeat_id,
        'text': answer, 'source': 'Gemini API',
    }
    CALL_LOG.append(record)
    return record

def show_response(record):
    print('Model:', record['model'], '| Temperature:', record['params']['temperature'])
    print(record['text'])


# 4. Experiment
## A — Summarization
### Bước 6 — Prompt ngắn và prompt có ràng buộc
Cùng incident, cùng tham số thấp; chỉ thay prompt. Kiểm tra output bằng text gốc T01, không chỉ dựa vào độ trôi chảy.


In [10]:
SUMMARY_SHORT = 'Tóm tắt sự cố này.'
SUMMARY_CLEAR = (
    'Bạn hỗ trợ kỹ thuật viên vận hành. Chỉ dùng thông tin trong incident. '
    'Tóm tắt 3–5 bullet bao gồm symptom, impact, timeline, action đã thực hiện và unresolved issue. '
    'Giữ số liệu và mốc thời gian quan trọng. Phân biệt nghi ngờ với nguyên nhân đã xác nhận; '
    'không thêm nguyên nhân, hành động hoặc kết quả không có trong input.')
# THỰC HÀNH: sửa SUMMARY_CLEAR để cải thiện một điểm cụ thể; ghi thay đổi.
a_short = call_llm('summary','T01',SUMMARY_SHORT)
a_clear = call_llm('summary','T01',SUMMARY_CLEAR)
show_response(a_short); print('\n--- Prompt rõ ---'); show_response(a_clear)


Model: gemini-3.5-flash-lite | Temperature: 0.1
Dưới đây là tóm tắt ngắn gọn về sự cố tại Site HN01:

* **Thời gian bắt đầu:** 20:05 ngày 10/01/2026.
* **Mức độ:** Nghiêm trọng (Severity cao).
* **Ảnh hưởng:** Khoảng 120 người dùng mất kết nối dịch vụ, không ping được gateway.
* **Diễn biến xử lý:**
    * **20:15:** Kỹ thuật viên kiểm tra cáp và nguồn tại chỗ nhưng chưa phát hiện bất thường.
    * **20:25:** Vẫn không truy cập được gateway, đã mở ticket yêu cầu nhà cung cấp đường truyền hỗ trợ.
    * **20:45:** Sự cố vẫn chưa được khắc phục, đang chờ phản hồi và xác nhận nguyên nhân từ nhà cung cấp (nghi ngờ lỗi do đường truyền phía nhà mạng).

--- Prompt rõ ---
Model: gemini-3.5-flash-lite | Temperature: 0.1
Dựa trên thông tin từ incident, tóm tắt các điểm chính như sau:

*   **Symptom & Impact:** Site HN01 mất kết nối từ 20:05 ngày 10/01/2026 (severity cao), khiến khoảng 120 người dùng không truy cập được dịch vụ và ping tới gateway thất bại.
*   **Timeline & Action:** 
    *   20:15

### Bước 7 — Đối chiếu fact
Các fact cần xem: HN01; bắt đầu 20:05 ngày 10/01; khoảng 120 người dùng; ping gateway thất bại; kiểm tra cáp/nguồn lúc 20:15 chưa thấy bất thường; mở ticket nhà cung cấp; nguyên nhân chưa xác nhận; 20:45 chưa khôi phục.

**THỰC HÀNH — không tự động gán Good vì output đủ bullet:**

| Output | Fact quan trọng bị thiếu | Fact bị thêm/đổi | Hypothesis thành conclusion? | Good/Partial/Poor + lý do |
|---|---|---|---|---|
| Prompt ngắn | … | … | … | … |
| Prompt rõ | … | … | … | … |

Tóm tắt có thể gộp timeline/action vào một bullet; không bắt buộc 5 bullet nếu đủ nội dung.


## B — Ticket Classification
### Bước 8 — Phân loại toàn bộ 15 ticket
Model phải trả đúng một label. Strict validation không tự tìm label trong câu dài để che lỗi format. Accuracy strict tính output ngoài label set là sai; accuracy trên output hợp lệ chỉ là chỉ số phụ.


In [11]:
CLASSIFY_PROMPT = (
    'Phân loại ticket theo nhóm xử lý chính. Chỉ trả đúng một label, không giải thích: '
    'Connectivity / Performance / Hardware / Configuration / Other. '
    'Connectivity: mất hoặc không thiết lập được kết nối. '
    'Performance: vẫn kết nối nhưng chậm, độ trễ/jitter/throughput bất thường. '
    'Hardware: có bằng chứng trực tiếp linh kiện hỏng. '
    'Configuration: có bằng chứng sai lệch cấu hình. '
    'Other: yêu cầu khác hoặc thiếu bằng chứng xác định nhóm chính. '
    'Không suy đoán nguyên nhân chỉ từ triệu chứng chung.')

# Gộp ticket trong một request để giảm quota; không gửi nhãn đáp án.
BATCH_PROMPT = CLASSIFY_PROMPT.replace('Chỉ trả đúng một label, không giải thích:', 'Các label được phép:') + (
    ' Phân loại từng ticket độc lập trong danh sách input. Trả JSON array thuần, '
    'mỗi phần tử đúng hai keys ticket_id và label. Giữ nguyên ID, mỗi ID đúng một lần; không markdown.')
batch_input = json.dumps([{'ticket_id':i['ticket_id'],'text':i['text']} for i in INCIDENTS],ensure_ascii=False)
batch_record = call_llm('classification_batch','ALL',BATCH_PROMPT,text_override=batch_input)
show_response(batch_record)
try:
    batch_output = json.loads(batch_record['text'])
except json.JSONDecodeError:
    batch_output = []
    print('Batch không phải JSON thuần; giữ raw output và ghi nhận lỗi format trước khi chỉnh prompt.')
if not isinstance(batch_output,list): batch_output = []
classification_rows = []
for item in INCIDENTS:
    matches = [v for v in batch_output if isinstance(v,dict) and v.get('ticket_id')==item['ticket_id']]
    pred = matches[0].get('label') if len(matches)==1 else None
    valid = (len(matches)==1 and set(matches[0])=={'ticket_id','label'}
             and isinstance(pred,str) and pred in LABELS)
    classification_rows.append({'ticket_id':item['ticket_id'],'ground_truth':item['label'],
        'prediction':pred,'valid_label':valid,'correct':valid and pred==item['label'],
        'ambiguous':item['ambiguous'],'source':batch_record['source']})
expected_ids = {i['ticket_id'] for i in INCIDENTS}
extra_rows = [v for v in batch_output if not isinstance(v,dict) or v.get('ticket_id') not in expected_ids]
print('Số phần tử thừa/không hợp lệ:',len(extra_rows),'| Số phần tử trả về:',len(batch_output))
classification_results = pd.DataFrame(classification_rows)
display(classification_results)
classification_accuracy = float(classification_results['correct'].mean())
classification_valid_rate = float(classification_results['valid_label'].mean())
print('Tỷ lệ output hợp lệ:', classification_valid_rate)
print('Accuracy strict trên bộ nhỏ:', classification_accuracy)
print('Các mẫu cần xem:')
display(classification_results.query('not correct or ambiguous or not valid_label'))


Model: gemini-3.5-flash-lite | Temperature: 0.1
[
  {"ticket_id": "T01", "label": "Connectivity"},
  {"ticket_id": "T02", "label": "Performance"},
  {"ticket_id": "T03", "label": "Hardware"},
  {"ticket_id": "T04", "label": "Configuration"},
  {"ticket_id": "T05", "label": "Other"},
  {"ticket_id": "T06", "label": "Connectivity"},
  {"ticket_id": "T07", "label": "Performance"},
  {"ticket_id": "T08", "label": "Hardware"},
  {"ticket_id": "T09", "label": "Hardware"},
  {"ticket_id": "T10", "label": "Configuration"},
  {"ticket_id": "T11", "label": "Other"},
  {"ticket_id": "T12", "label": "Connectivity"},
  {"ticket_id": "T13", "label": "Performance"},
  {"ticket_id": "T14", "label": "Other"},
  {"ticket_id": "T15", "label": "Other"}
]
Số phần tử thừa/không hợp lệ: 0 | Số phần tử trả về: 15


,ticket_id,ground_truth,prediction,valid_label,correct,ambiguous,source
0,T01,Connectivity,Connectivity,True,True,False,Gemini API
1,T02,Performance,Performance,True,True,False,Gemini API
2,T03,Hardware,Hardware,True,True,False,Gemini API
3,T04,Configuration,Configuration,True,True,False,Gemini API
4,T05,Other,Other,True,True,False,Gemini API
5,T06,Connectivity,Connectivity,True,True,False,Gemini API
6,T07,Performance,Performance,True,True,False,Gemini API
7,T08,Hardware,Hardware,True,True,False,Gemini API
8,T09,Hardware,Hardware,True,True,False,Gemini API
9,T10,Configuration,Configuration,True,True,False,Gemini API


Tỷ lệ output hợp lệ: 1.0
Accuracy strict trên bộ nhỏ: 1.0
Các mẫu cần xem:


,ticket_id,ground_truth,prediction,valid_label,correct,ambiguous,source
13,T14,Other,Other,True,True,True,Gemini API
14,T15,Other,Other,True,True,True,Gemini API


**THỰC HÀNH:** chọn một mẫu sai/mơ hồ. So sánh symptom với quy tắc nhãn; model sai, nhãn tham chiếu chưa rõ hay task chưa đủ dữ liệu? Ghi ticket_id, input, output và giải thích. …

Không sửa ground truth sau khi xem output chỉ để tăng accuracy. Nếu thấy nhãn hợp lý khác, ghi giới hạn và đề xuất quy tắc cho lần đánh giá sau.


## C — Text → JSON
### Bước 9 — Trích xuất T09: cố ý thiếu site, severity, time và cause
Schema: đúng 5 keys `site_id, symptom, severity, time, suspected_cause`. Mỗi field là string hoặc null. Severity nếu có: low/medium/high/critical. Thiếu dữ kiện → null; không tự biến hư hỏng quan sát được thành nguyên nhân.


In [14]:
JSON_SHORT = 'Chuyển ticket thành JSON gồm site_id, symptom, severity, time, suspected_cause, trong một dòng'
JSON_CLEAR = (
    'Trích thông tin chỉ từ incident, trả đúng một JSON object, không markdown và không giải thích. '
    'Đúng 5 keys: site_id, symptom, severity, time, suspected_cause. '
    'Mỗi field là string hoặc null. Severity chỉ low, medium, high, critical hoặc null. '
    'Field không được cung cấp phải null. suspected_cause chỉ ghi khi input nêu giả thuyết '
    'hoặc nguyên nhân, giữ rõ mức độ nghi ngờ; không tự suy diễn. '
    'Chân cổng bị gãy là symptom, không đủ bằng chứng xác định vì sao bị gãy.')
c_short = call_llm('json','T09',JSON_SHORT)
c_clear = call_llm('json','T09',JSON_CLEAR)
show_response(c_short); print('\n--- Prompt rõ ---'); show_response(c_clear)


Model: gemini-3.5-flash-lite | Temperature: 0.1
`{"site_id": null, "symptom": "Cổng switch có chân tiếp xúc bị gãy", "severity": null, "time": null, "suspected_cause": null}`

--- Prompt rõ ---
Model: gemini-3.5-flash-lite | Temperature: 0.1
{"site_id": null, "symptom": "Chân tiếp xúc cổng switch bị gãy", "severity": null, "time": null, "suspected_cause": null}


In [15]:
SCHEMA_KEYS = {'site_id','symptom','severity','time','suspected_cause'}
def validate_json(text):
    errors = []
    try: obj = json.loads(text)
    except json.JSONDecodeError as e:
        return {'parse_ok':False,'schema_ok':False,'errors':[f'JSON parse lỗi tại ký tự {e.pos}'],'object':None}
    if not isinstance(obj,dict):
        return {'parse_ok':True,'schema_ok':False,'errors':['Root phải là object'],'object':obj}
    if set(obj) != SCHEMA_KEYS: errors.append('Keys thiếu/thừa so với schema.')
    for key, value in obj.items():
        if value is not None and not isinstance(value,str): errors.append(f'{key} phải string hoặc null.')
    if obj.get('severity') not in (None,'low','medium','high','critical'):
        errors.append('severity ngoài enum.')
    return {'parse_ok':True,'schema_ok':not errors,'errors':errors,'object':obj}

json_checks = []
for name, record in [('short',c_short),('clear',c_clear)]:
    check = validate_json(record['text'])
    json_checks.append({'prompt':name,'source':record['source'],**check})
display(pd.DataFrame(json_checks))
# T09 có 4 field thiếu; kiểm tra nội dung độc lập với parse/schema.
obj = json_checks[1]['object']
if isinstance(obj,dict):
    missing_field_errors = [key for key in ['site_id','severity','time','suspected_cause'] if obj.get(key) is not None]
    print('Field thiếu nhưng model đã điền:', missing_field_errors)


,prompt,source,parse_ok,schema_ok,errors,object
0,short,Gemini API,False,False,[JSON parse lỗi tại ký tự 0],None
1,clear,Gemini API,True,True,[],"{'site_id': None, 'symptom': 'Chân tiếp xúc cổ..."


Field thiếu nhưng model đã điền: []


**THỰC HÀNH:** JSON parse được có đồng nghĩa đúng sự thật không? Tự đối chiếu symptom với T09. Chọn một ticket khác, chỉnh prompt hoặc gọi thêm để kiểm tra logic của bạn. Nếu sửa validator, thêm một ví dụ hợp lệ và một ví dụ sai.

Không tự bỏ code fence rồi báo rằng model đã tuân thủ “JSON thuần”. Có thể sửa output cho nghiệp vụ, nhưng phải ghi rõ raw output ban đầu sai format.


## D — Recommendation
### Bước 10 — Đề xuất 3 bước kiểm tra tiếp theo
Đây là nội dung hỗ trợ kỹ thuật viên, không phải lệnh được thực thi. Tách fact/hypothesis; chỉ yêu cầu thao tác đọc/thu thập thông tin; thiếu dữ liệu phải nói rõ. Không đề xuất hoặc mô phỏng reboot, thay cấu hình, shutdown, rollback hay thao tác write.


In [16]:
RECOMMEND_PROMPT = (
    'Đề xuất đúng 3 bước kiểm tra tiếp theo cho kỹ thuật viên, dựa trên incident. '
    'Nêu fact đã biết và hypothesis cần kiểm chứng. Nếu chưa đủ dữ liệu xác định nguyên nhân, '
    'nói rõ chưa đủ thông tin. Chỉ đề xuất quan sát, đọc trạng thái/log hoặc hỏi thêm. '
    'Không ra lệnh, không mô phỏng đã thực hiện, không đề xuất thay cấu hình, reboot, '
    'shutdown hoặc thay thiết bị. Với mỗi bước, nói thông tin cần thu thập. '
    'Nêu kết quả cần kỹ thuật viên review trước khi quyết định.')
d_clear = call_llm('recommendation','T01',RECOMMEND_PROMPT)
show_response(d_clear)


Model: gemini-3.5-flash-lite | Temperature: 0.1
**Bước 1:**

*   **Fact đã biết:** Site HN01 mất kết nối từ 20:05, ping tới gateway thất bại, kiểm tra vật lý (cáp/nguồn) tại chỗ bình thường, đang chờ nhà cung cấp đường truyền phản hồi.
*   **Hypothesis cần kiểm chứng:** Đường truyền WAN chính đang bị đứt/mất tín hiệu quang hoặc nhà cung cấp đã hạ đường truyền mà chưa thông báo.
*   **Thông tin cần thu thập:** Đọc trạng thái đèn tín hiệu (Link/Power/Alarm) trên thiết bị truyền dẫn (Modem/ONT/Router biên) kết nối trực tiếp với nhà mạng; hoặc đọc log trạng thái cổng WAN (Interface status, BGP/OSPF neighbor state nếu có) trên router biên của site HN01.
*   **Kết quả cần review:** Trạng thái vật lý/logical của cổng WAN trên thiết bị biên (up/down) và phản hồi hiện tại từ ticket của nhà cung cấp đường truyền để xác định lỗi thuộc về hạ tầng nhà mạng hay thiết bị nội bộ.

**Bước 2:**

*   **Fact đã biết:** Khoảng 120 người dùng không truy cập được dịch vụ, ping gateway thất bại, kỹ thuật viên

**THỰC HÀNH — đánh giá thủ công:**
- Useful? Ba bước có cụ thể và giúp giảm điều chưa biết không? …
- Safe theo phạm vi lab? Có thao tác write/change hoặc tuyên bố đã thực hiện không? …
- Needs human review? Ai review và trước quyết định nào? …
- Có biến hypothesis thành fact hoặc thêm thông tin không có trong ticket không? …

Một bước kiểm tra hữu ích không chứng minh nguyên nhân đã đúng.


## Prompt / Parameter / Consistency
### Bước 11 — Lặp cùng prompt rõ để kiểm tra consistency
So sánh hai lần với cùng prompt/param/input cho A/C/D. B đã có một nhãn cho mỗi ticket trong một request gộp; bước tiếp theo lặp T14. Exact match chỉ là chỉ số format/text; hai câu khác từ vẫn có thể nhất quán về fact và ý nghĩa.

Các lần lặp đều gọi API mới. So sánh ngữ nghĩa và fact, không chỉ exact match.


In [17]:
a_repeat = call_llm('summary','T01',SUMMARY_CLEAR,repeat_id=1)
c_repeat = call_llm('json','T09',JSON_CLEAR,repeat_id=1)
d_repeat = call_llm('recommendation','T01',RECOMMEND_PROMPT,repeat_id=1)
consistency_examples = [('Summarize',a_clear,a_repeat),('Text → JSON',c_clear,c_repeat),('Recommend',d_clear,d_repeat)]
for task, first, second in consistency_examples:
    print('\nTask:',task,'| exact text match:',first['text']==second['text'])
    show_response(second)



Task: Summarize | exact text match: False
Model: gemini-3.5-flash-lite | Temperature: 0.1
Dưới đây là tóm tắt thông tin sự cố theo yêu cầu:

* **Symptom & Impact:** Site HN01 mất kết nối từ 20:05 ngày 10/01/2026 (severity cao), khiến khoảng 120 người dùng không thể truy cập dịch vụ và ping tới gateway thất bại.
* **Timeline & Actions:** 
  * **20:15:** Kỹ thuật viên kiểm tra cáp và nguồn, ghi nhận chưa có dấu hiệu bất thường.
  * **20:25:** Vẫn không truy cập được gateway, kỹ thuật viên đã mở ticket với nhà cung cấp đường truyền.
* **Root Cause (Nghi ngờ vs Đã xác nhận):** Đường truyền phía nhà cung cấp được **nghi ngờ** là nguyên nhân, nhưng **chưa có log hoặc xác nhận nguyên nhân**.
* **Unresolved Issue:** Đến 20:45 dịch vụ vẫn chưa được khôi phục và đang chờ phản hồi từ nhà cung cấp.

Task: Text → JSON | exact text match: False
Model: gemini-3.5-flash-lite | Temperature: 0.1
{"site_id": null, "symptom": "chân cổng bị gãy", "severity": null, "time": null, "suspected_cause": null}

T

### Bước 12 — Temperature thấp ↔ cao, giữ nguyên prompt/input
Lặp 3 lần mỗi mức 0.1 và 0.9 trên ticket mơ hồ T14. Không đổi prompt, model hoặc token limit cùng lúc. Temperature cao không đảm bảo output sai hơn; sáu lần gọi trên một case chỉ là phép thử nhỏ. Các tham số không gửi giữ mặc định model.


In [19]:
parameter_rows = []
for value in [LOW_VALUE, HIGH_VALUE]:
    for repeat in range(3):
        r = call_llm('recommendation', 'T14', RECOMMEND_PROMPT, value=value, repeat_id=repeat)
        prediction = r['text'].strip()
        parameter_rows.append({'parameter':'temperature', 'value':value, 'repeat':repeat,
            'output':prediction, 'valid_label':prediction in LABELS, 'source':r['source']})
parameter_results = pd.DataFrame(parameter_rows)
display(parameter_results)
display(parameter_results.groupby('value').agg(
    distinct_outputs=('output','nunique'), valid_rate=('valid_label','mean')))


,parameter,value,repeat,output,valid_label,source
0,temperature,0.1,0,"Dựa trên incident hiện tại, dữ liệu chưa đủ để...",False,Gemini API
1,temperature,0.1,1,**1. Kiểm tra trạng thái thiết bị và log hệ th...,False,Gemini API
2,temperature,0.1,2,Chưa đủ thông tin để xác định nguyên nhân. \n\...,False,Gemini API
3,temperature,0.9,0,"Dựa trên incident tại site HCM14, do chưa đủ d...",False,Gemini API
4,temperature,0.9,1,"Dựa trên incident hiện tại, dữ liệu chưa đủ để...",False,Gemini API
5,temperature,0.9,2,"Dựa trên incident hiện tại, dữ liệu chưa đủ để...",False,Gemini API


,distinct_outputs,valid_rate
value,,
0.1,3,0.0
0.9,3,0.0


In [22]:
# Thí nghiệm so sánh mức độ ảnh hưởng của Temperature (Thấp vs Cao) trên Task D (Recommendation)
ticket_id_test = 'T14'

print(f"--- Đang gọi thử nghiệm với Temperature Thấp ({LOW_VALUE}) ---")
rec_low = call_llm('recommendation_test', ticket_id_test, RECOMMEND_PROMPT, value=LOW_VALUE, repeat_id=99)

print(f"\n--- Đang gọi thử nghiệm với Temperature Cao ({HIGH_VALUE}) ---")
rec_high = call_llm('recommendation_test', ticket_id_test, RECOMMEND_PROMPT, value=HIGH_VALUE, repeat_id=99)

print("\n========================================================================")
print(f"[KẾT QUẢ ĐỐI CHIẾU] Sự khác biệt khi thay đổi Temperature trên Ticket {ticket_id_test}:")
print("========================================================================\n")

print(f"=== TEMPERATURE THẤP ({LOW_VALUE}) - Thường tập trung, chặt chẽ và nhất quán ===")
print(rec_low['text'])
print("\n" + "="*40 + "\n")
print(f"=== TEMPERATURE CAO ({HIGH_VALUE}) - Thường đa dạng từ ngữ, phong phú và ngẫu nhiên ===")
print(rec_high['text'])


--- Đang gọi thử nghiệm với Temperature Thấp (0.1) ---

--- Đang gọi thử nghiệm với Temperature Cao (0.9) ---

[KẾT QUẢ ĐỐI CHIẾU] Sự khác biệt khi thay đổi Temperature trên Ticket T14:

=== TEMPERATURE THẤP (0.1) - Thường tập trung, chặt chẽ và nhất quán ===
Dựa trên incident tại site HCM14, dưới đây là phân tích và 3 bước kiểm tra tiếp theo dành cho kỹ thuật viên:

*   **Fact đã biết:** Site HCM14 có hiện tượng chập chờn (timeout xen lẫn phản hồi chậm) sau một lần bảo trì.
*   **Hypothesis cần kiểm chứng:** 
    *   Giả thuyết 1: Thiết bị mạng/máy chủ tại site đang gặp vấn đề về quá tải tài nguyên (CPU/RAM/Buffer) hoặc lỗi kết nối vật lý (cáp quang, port) sau bảo trì.
    *   Giả thuyết 2: Có thay đổi về cấu hình (routing, firewall, QoS) hoặc phần cứng trong đợt bảo trì gây ra xung đột hoặc nghẽn cổ chai.
*   **Tình trạng dữ liệu:** *Chưa đủ thông tin* để xác định nguyên nhân gốc rễ do thiếu log và chưa rõ lịch sử thay đổi.

---

### 3 bước kiểm tra tiếp theo:

**Bước 1: Thu thập thô

**THỰC HÀNH:**
1. Prompt rõ cải thiện điều gì so với prompt ngắn ở A/C? Có điểm nào vẫn sai? …
2. Tham số nào thật sự được gửi? Có khác biệt giữa 3 lần lặp không? …
3. Output khác chữ hay khác nhãn/fact/ý nghĩa? …
4. Bạn có đủ bằng chứng nói model ổn định không? Giới hạn của phép thử? …


## Missing-data hallucination test
### Bước 13 — Cố ý xóa dữ kiện và hỏi nguyên nhân
Tạo input rút gọn từ T01: bỏ site, mốc thời gian, phạm vi, kết quả kiểm tra và giả thuyết nhà cung cấp. Chỉ giữ mô tả mất kết nối chung. Hai prompt khác nhau trên cùng input thiếu: prompt dẫn dắt và prompt yêu cầu bằng chứng.

Mục tiêu không phải ép model sai; quan sát model có nói thiếu dữ liệu, trình bày giả thuyết hay khẳng định không có căn cứ. Trước khi gọi model, viết bạn có thể kết luận điều gì từ input này.


In [23]:
MISSING_INPUT = 'Một số người dùng báo không truy cập được dịch vụ. Chưa cung cấp kết quả kiểm tra hoặc log.'
LEADING_PROMPT = 'Hãy cho biết chính xác nguyên nhân gây sự cố này và thiết bị nào bị hỏng.'
BOUNDED_PROMPT = (
    'Chỉ dựa trên incident, có đủ bằng chứng kết luận nguyên nhân không? '
    'Nếu không đủ, nói rõ chưa đủ thông tin, không tự điền site, thời gian hoặc thiết bị. '
    'Liệt kê dữ liệu cần bổ sung; nếu nêu giả thuyết, đánh dấu là chưa được xác nhận.')
h_leading = call_llm('missing_cause','T01',LEADING_PROMPT,text_override=MISSING_INPUT)
h_bounded = call_llm('missing_cause','T01',BOUNDED_PROMPT,text_override=MISSING_INPUT)
print('Input đã rút gọn:',MISSING_INPUT)
show_response(h_leading); print('\n--- Có ràng buộc ---'); show_response(h_bounded)


Input đã rút gọn: Một số người dùng báo không truy cập được dịch vụ. Chưa cung cấp kết quả kiểm tra hoặc log.
Model: gemini-3.5-flash-lite | Temperature: 0.1
Dựa trên thông tin bạn cung cấp trong `<incident>`, **không thể xác định chính xác nguyên nhân gây sự cố cũng như thiết bị nào bị hỏng**. 

Lý do là vì dữ liệu hiện tại quá ít (*"Chưa cung cấp kết quả kiểm tra hoặc log"* và *"Một số người dùng báo không truy cập được dịch vụ"*). Triệu chứng "không truy cập được dịch vụ" chỉ là phần nổi của tảng băng chìm và có thể do rất nhiều nguyên nhân khác nhau gây ra.

Để tìm ra chính xác nguyên nhân và thiết bị hỏng, cần phải thực hiện các bước chẩn đoán tiếp theo, chẳng hạn như:
1. **Kiểm tra hạ tầng mạng:** Router, Switch, Firewall, cáp mạng có bị đứt hoặc lỗi phần cứng không?
2. **Kiểm tra máy chủ (Server) / Dịch vụ:** Server có bị sập nguồn, quá tải CPU/RAM, hay dịch vụ (web, database...) bị dừng đột ngột không?
3. **Kiểm tra DNS hoặc Caching:** Lỗi phân giải tên miền hay cấu hình sai?
4

**THỰC HÀNH — ghi nhận:**
- Model từ chối/nói thiếu dữ liệu, hay tự khẳng định? …
- Claim nào không được input hỗ trợ? Trích ngắn câu output và đối chiếu. …
- Nếu model chỉ nêu khả năng và đánh dấu chưa xác nhận, có phải hallucination không? Giải thích. …
- Prompt rõ có loại bỏ hoàn toàn rủi ro không, hay chỉ cải thiện case đã thử? …

## Failure case bắt buộc
Có thể chọn sai fact ở A, sai/không hợp lệ ở B, JSON sai schema/nội dung ở C, recommendation vượt phạm vi ở D hoặc khẳng định nguyên nhân trong test thiếu dữ liệu.

| Nội dung | Nhóm điền |
|---|---|
| Task / ticket / nguồn output | … |
| Input và prompt thực tế | … |
| Output lỗi (trích đúng) | … |
| Bằng chứng lỗi từ input/rubric | … |
| Hậu quả nếu dùng thật | … |
| Biện pháp: kiểm thử, constraint, human review hay không dùng? | … |

Nếu chưa thấy lỗi trên model thật, mở rộng một case thiếu dữ liệu/mơ hồ và ghi trung thực kết quả. Không tự dựng output rồi gắn cho model.


# 5. Evaluation
### Bước 14 — Tổng hợp bằng chứng và điền bảng 4 task
Bảng classification/JSON có kiểm tra tự động nhưng không thay thế việc đối chiếu sự thật. A/D cần đánh giá bằng mắt. Giảng viên cung cấp rubric bên dưới; không dùng chính LLM đang thử làm nguồn chấm duy nhất.

| Tiêu chí | Gợi ý cách đọc |
|---|---|
| Quality | Good: đáp ứng chính và không sai fact trọng yếu; Partial: thiếu/không đầy đủ; Poor: sai fact trọng yếu hoặc output không dùng được |
| Consistency | So sánh những lần lặp đã có; phân biệt exact match với nhất quán ngữ nghĩa; chưa đủ lần thử → ghi chưa đủ bằng chứng |
| Hallucination risk | Nhận xét dựa trên claim được/không được input hỗ trợ, không gán “0%” chỉ từ vài mẫu |
| Business impact | Hậu quả của sai sót: chuyển sai nhóm, thêm dữ liệu sai, kết luận sai nguyên nhân… |
| Use? | Yes with review / Limited pilot / No; ghi điều kiện |

**THỰC HÀNH:** thay các ô “Chưa đánh giá” trong cell dưới bằng nhận xét có số liệu/ticket hoặc call tham chiếu. Không tự động điền Good từ accuracy.


In [24]:
evaluation = pd.DataFrame([
 {'Task':task,'Quality':'Chưa đánh giá','Consistency':'Chưa đánh giá',
  'Hallucination risk':'Chưa đánh giá','Business impact':'Chưa đánh giá',
  'Use?':'Chưa đánh giá','Evidence/source':'Chưa đánh giá'}
 for task in ['Summarize','Classify','Text → JSON','Recommend']
])
# THỰC HÀNH: điền nhận xét, ví dụ:
# evaluation.loc[1, 'Quality'] = '... kèm tỷ lệ đúng và mẫu cần xem'
display(evaluation)
print('Accuracy classification trên bộ mẫu:',classification_accuracy)
print('JSON prompt rõ:',validate_json(c_clear['text'])['schema_ok'])
print('Nguồn output trong phiên:',dict(Counter(r['source'] for r in CALL_LOG)))


,Task,Quality,Consistency,Hallucination risk,Business impact,Use?,Evidence/source
0,Summarize,Chưa đánh giá,Chưa đánh giá,Chưa đánh giá,Chưa đánh giá,Chưa đánh giá,Chưa đánh giá
1,Classify,Chưa đánh giá,Chưa đánh giá,Chưa đánh giá,Chưa đánh giá,Chưa đánh giá,Chưa đánh giá
2,Text → JSON,Chưa đánh giá,Chưa đánh giá,Chưa đánh giá,Chưa đánh giá,Chưa đánh giá,Chưa đánh giá
3,Recommend,Chưa đánh giá,Chưa đánh giá,Chưa đánh giá,Chưa đánh giá,Chưa đánh giá,Chưa đánh giá


Accuracy classification trên bộ mẫu: 1.0
JSON prompt rõ: True
Nguồn output trong phiên: {'Gemini API': 26}


# 6. Business Reflection
### Bước 15 — Quyết định có nên dùng
**Viết 100–150 từ dựa trên kết quả thực tế:**
- Task nào nên thử dùng, với điều kiện gì? …
- Model sai hoặc không chắc ở đâu? …
- Nếu dùng thật, lỗi đó gây hậu quả gì? …
- Có phương pháp đơn giản hơn AI không? …
- Human review cần ở đâu, và review thông tin gì? …

### Bước 16 — Ba nguyên tắc cá nhân: khi nào không nên dùng LLM
Viết nguyên tắc cụ thể theo dạng **điều kiện → quyết định → lý do**. Không chỉ viết “LLM có thể sai”.
1. …
2. …
3. …

**Câu hỏi gợi ý:** thiếu dữ liệu, không có cách kiểm chứng, hay tác động nghiệp vụ vượt quá mức chấp nhận được có thay đổi quyết định của bạn không?


### Bước 17 — Lưu và nộp
Lưu prompt, temperature, raw output và bảng đánh giá để đối chiếu. Không lưu credential. Cell dưới xuất CSV/JSON; sinh viên chỉ cần nộp notebook có outputs và phần nhận xét đã điền. Trong Colab: **File → Download → Download .ipynb**.


In [25]:
if evaluation.eq('Chưa đánh giá').any().any():
    print('Bảng evaluation còn ô chưa điền; hoàn thiện trước khi nộp.')
classification_results.to_csv(OUTPUT_DIR/'classification_results.csv',index=False)
evaluation.to_csv(OUTPUT_DIR/'evaluation_4_tasks.csv',index=False)
(OUTPUT_DIR/'call_log.json').write_text(json.dumps(CALL_LOG,ensure_ascii=False,indent=2),encoding='utf-8')
(OUTPUT_DIR/'run_config.json').write_text(json.dumps({
    'model':MODEL_NAME,'api_url':API_URL,'sampling_param':'temperature',
    'low':LOW_VALUE,'high':HIGH_VALUE,'max_output_tokens':MAX_OUTPUT_TOKENS,
    'token_limit_field':'maxOutputTokens','sources':dict(Counter(r['source'] for r in CALL_LOG)),
    'simulated_dataset':True},ensure_ascii=False,indent=2),encoding='utf-8')
if parameter_rows:
    pd.DataFrame(parameter_rows).to_csv(OUTPUT_DIR/'parameter_results.csv',index=False)
print('Đã lưu các bảng và raw output tại:', OUTPUT_DIR.resolve())
print('Nộp duy nhất notebook đã lưu outputs: File → Download → Download .ipynb trong Colab.')


Bảng evaluation còn ô chưa điền; hoàn thiện trước khi nộp.
Đã lưu các bảng và raw output tại: /content/lab1_outputs
Nộp duy nhất notebook đã lưu outputs: File → Download → Download .ipynb trong Colab.


## Checklist hoàn thành
- [ ] Hoàn thiện Business Context và đọc dữ liệu thiếu/mơ hồ.
- [ ] Có output A/B/C/D; ghi rõ model, backend và nguồn từng output.
- [ ] Classification chạy đủ 15 ticket; ghi tỷ lệ nhãn hợp lệ và accuracy/đối chiếu thủ công.
- [ ] JSON có kiểm tra parse, schema và nội dung field thiếu.
- [ ] Thử prompt ngắn → rõ và tham số thấp → cao trên backend hỗ trợ.
- [ ] Có missing-data test và ít nhất một failure case có bằng chứng.
- [ ] Bảng evaluation đủ 4 task, reflection và 3 nguyên tắc cá nhân.
- [ ] Lưu notebook có output và tải kết quả; không có key trong code/output.

Không kết luận một model khả thi cho nghiệp vụ chỉ từ 15 ticket mô phỏng.

**Nộp:** `Module3_Lab1_<ten_nhom>.ipynb` có output. Không bắt buộc kết quả tốt; quan trọng là thấy lỗi và đưa ra quyết định có căn cứ.

## Nguồn
Thiết kế theo slide “AI Engineer – Module 3, Lab Bài 1: LLM có làm được việc này không?” do giảng viên cung cấp: 6 phần, 4 task, 90 phút. Dataset do tác giả notebook mô phỏng cho mục đích giảng dạy.

Tài liệu: [Gemini generateContent REST](https://ai.google.dev/api/generate-content?hl=en), [API keys](https://ai.google.dev/gemini-api/docs/api-key), [free tier pricing](https://ai.google.dev/gemini-api/docs/pricing), [rate limits](https://ai.google.dev/gemini-api/docs/rate-limits). Kiểm tra model/quota trước khi phát lớp. Phiên bản notebook dành cho Gemini API từ Google AI Studio; không yêu cầu mua API hoặc GPU.

Đối chiếu ngày 07/10/2026: [model 3.5 Flash-Lite](https://ai.google.dev/gemini-api/docs/models/gemini-3.5-flash-lite), [models.get](https://ai.google.dev/api/models), [generateContent và GenerationConfig](https://ai.google.dev/api/generate-content), [pricing](https://ai.google.dev/gemini-api/docs/pricing). Chưa xác nhận quota/quyền của tài khoản sinh viên; chạy Bước 4b và 5b để kiểm tra.
